# IMAPA Tutorial

This tutorial explains how to use the **IMAPA** (Intermittent Multiple Aggregation Prediction Algorithm) forecasting model. IMAPA is designed for **intermittent demand** time series — data where many observations are zero or near zero and demand occurs sporadically.

This implementation mirrors the high-level algorithm used by StatsForecast.IMAPA, while integrating tightly with this library's JAX architecture.

In [ ]:
import sys
import os

# Add parent directory to path so we can import the library modules
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import numpy as np

from imapa import IMAPA
from conformal_intervals import ConformalIntervals

# Model Overview + Example Walkthrough

IMAPA improves forecasting for intermittent series through three steps:

### Step 1. Aggregate at multiple temporal resolutions ("levels")

Example: daily data → `[1, 0, 0, 3, 0, 0, 2, ...]`

- Level 2: sum every 2 days → `[1, 3, 2, ...]`
- Level 3: sum every 3 days → `[1, 3, 2, ...]`

The **purpose** is to reduce data sparsity. Different levels capture different information — fine levels reflect short-term variation while higher levels reveal the overall demand rate, smoothing out noise.

### Step 2. Fit Simple Exponential Smoothing (SES) at each level

SES is a lightweight forecasting method:

*Forecast = α × (last observation) + (1 − α) × (last forecast)*

Each aggregation level has its own SES model with its own smoothing parameter α.

### Step 3. Back-map and average the forecasts

Each level's forecast is divided by its aggregation factor ("back-mapped") to return to the original time scale, then all levels are averaged to produce the final predictions.

**Back-mapping example:**

**Level 5:** forecast = 5 → back-map: 5.0 ÷ 5 = 1.0 units/day

| Day | Level 1 | Level 2 | Level 5 | Final (Average) |
|-----|---------|---------|---------|-----------------|
| 11  | 1.0     | 1.25    | 1.0     | **1.08**        |
| 12  | 0.9     | 1.25    | 1.0     | **1.05**        |
| 13  | 0.85    | 1.25    | 1.0     | **1.03**        |

# Math Overview

For each aggregation level:

Use SES forecasting for f:

$$\hat{y}_{t+1} = f(\hat{y}_{t-\tau}, \hat{y}_{t-2\tau}, ..., \hat{y}_{t-m\tau})$$

- $\hat{y}_{t+1}$: forecast for next time period
- $\tau$: Aggregation Level
- $f(\cdot)$: SES forecasting function: Forecast = α × (last\_observation) + (1-α) × (last\_forecast)

$$\text{Back-map: } \tilde{y}_{t+1}^{(\tau)} = \frac{\hat{y}_{t+1}^{(\tau)}}{\tau}$$

$$\text{Final: } \hat{y}_{t+1}^{\text{final}} = \frac{1}{K}\sum_{k=1}^{K} \tilde{y}_{t+1}^{(\tau_k)}$$

# When to Use
**Use when:**
* Demand is intermittent or sparse
* Many zero values exist
* Many missing values in our series

**Avoid when:**
* Series is dense and smooth
* Strong trend or seasonality dominates
* High frequency dynamics matter more than robustness

# API Contract

## Constructor

```python
IMAPA(
    alias: str = "IMAPA",
    conformal_params: Optional[ConformalIntervals] = None
)
```

- `alias`: Custom name for the model
- `conformal_params`: Configuration for conformal prediction intervals

## Fit Method

```python
fit(y: ArrayLike, X: Optional[ArrayLike] = None) -> IMAPA
```
- `y`: Historical time series data
- `X`: Placeholder for API consistency (ignored)

## Predict Method

```python
predict(h: int, level: Optional[List[int]] = None) -> Dict[str, ArrayLike]
```
- `h`: Forecast horizon
- `level`: Confidence levels (0–100) for conformal prediction intervals

## Predict in Sample

```python
predict_in_sample(level: Optional[List[int]] = None) -> Dict[str, ArrayLike]
```
Returns in-sample fitted values. Early fitted values may be NaN due to insufficient aggregation history.

## Forecast (Stateless)

```python
forecast(y, h, level=None, fitted=False) -> Dict[str, ArrayLike]
```
Stateless forecasting — fits and predicts without persisting model state.

# Examples

## Fit and Predict

SES produces **flat multi-step forecasts** — every future step equals the same value. We verify this property below.

In [ ]:
# Fit and Predict on intermittent data
y = jnp.asarray([0.0, 1.0, 0.0, 2.0, 0.0, 3.0, 0.0], dtype=jnp.float64)
m = IMAPA()
m.fit(y)

# The scalar value to be repeated is model_["mean"][0]
base = float(jnp.asarray(m.model_["mean"])[0])
print("Base forecast value:", base)

h = 5
out = m.predict(h=h, level=None)

print("Predictions:", out["mean"])
print("Shape:", out["mean"].shape)

# Verify all predictions equal the base (SES produces flat multi-step forecasts)
expected = jnp.full(h, base, dtype=jnp.float64)
assert jnp.allclose(out["mean"], expected, atol=1e-8), "predict mean should repeat the base value"

print("\nFit + predict: OK — forecasts are flat as expected for SES")

## Forecast (Stateless)

The `forecast()` method fits and predicts in a single call without storing any model state — useful for batch evaluation and cross-validation.

In [ ]:
# Stateless forecast
t = np.arange(36, dtype=np.float64)
y = jnp.asarray(5.0 + 0.1 * t, dtype=jnp.float64)

m = IMAPA()

# Without fitted values
out = m.forecast(y=y, h=4, level=None, fitted=False)
print("Forecast keys:", set(out.keys()))
print("Mean shape:", out["mean"].shape)

assert set(out.keys()) == {"mean"}
assert out["mean"].shape == (4,)

# With fitted values
out2 = m.forecast(y=y, h=3, level=None, fitted=True)
print("\nForecast with fitted — keys:", set(out2.keys()))
print("Mean shape:", out2["mean"].shape)
print("Fitted shape:", out2["fitted"].shape)

assert "mean" in out2 and "fitted" in out2
assert out2["mean"].shape == (3,)
assert out2["fitted"].shape == y.shape

print("\nStateless forecast: OK")

## Predict In Sample

`predict_in_sample()` returns the model's in-sample fitted values, which are useful for evaluating goodness of fit and computing residuals.

In [ ]:
# Predict in sample
y = jnp.asarray([0.0, 1.0, 0.0, 2.0, 0.0, 3.0, 0.0, 1.0, 0.0, 2.0], dtype=jnp.float64)
m = IMAPA()
m.fit(y)

res = m.predict_in_sample(level=None)
print("Fitted values:", res["fitted"])
print("Fitted shape:", res["fitted"].shape)

# With prediction intervals
res_pi = m.predict_in_sample(level=[80, 95])
print("\nKeys with intervals:", list(res_pi.keys()))

print("\nPredict in sample: OK")

# Requirements

- IMAPA is **univariate only** — it models a single time series
- Prediction intervals are **conformal** by design (no parametric distributional assumption)
- Early fitted values may contain `NaN`s due to insufficient aggregation history
- Point forecasts are designed to match StatsForecast.IMAPA behavior

# Edge Cases + Limitations

**Edge Cases:**
* Short series may not support all aggregation levels
* Early fitted values can be NaN
* Prediction intervals require sufficient data for conformity scoring

**Limitations:**
* Does not model seasonality
* Assumes historical forecast errors are representative
* Averaging across aggregations may smooth sharp dynamics